# 06 - Desempenho dos Grandes Clubes

Trajetória dos principais clubes brasileiros ao longo das temporadas: posição final, aproveitamento.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)

# Fix temporada 2020 (COVID): rodadas 28-38 jogadas em jan-fev/2021 pertencem a temporada 2020
mask_2020 = (df["data"].dt.year == 2021) & (df["rodata"] >= 28) & (df["data"] < "2021-03-01")
df.loc[mask_2020, "ano"] = 2020

df = df[df["ano"] >= 2003].copy()

In [2]:
def classificacao_final(df_season):
    teams = set(df_season["mandante"].unique()) | set(df_season["visitante"].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    
    for _, jogo in df_season.iterrows():
        m, v = jogo["mandante"], jogo["visitante"]
        gm, gv = jogo["mandante_Placar"], jogo["visitante_Placar"]
        if pd.isna(gm) or pd.isna(gv):
            continue
        gm, gv = int(gm), int(gv)
        saldo[m] += gm - gv
        saldo[v] += gv - gm
        if gm > gv:
            pontos[m] += 3; vitorias[m] += 1
        elif gm < gv:
            pontos[v] += 3; vitorias[v] += 1
        else:
            pontos[m] += 1; pontos[v] += 1
    
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    return {t: (i+1, pontos[t]) for i, t in enumerate(ranking)}

# Calcular posição final de todos os times em todas as temporadas
todas = []
for ano in sorted(df["ano"].unique()):
    df_s = df[df["ano"] == ano]
    df_s_clean = df_s.dropna(subset=["rodata"])
    if df_s_clean["rodata"].astype(int).max() < 30:
        continue
    classif = classificacao_final(df_s)
    for time, (pos, pts) in classif.items():
        todas.append({"ano": ano, "time": time, "posicao": pos, "pontos": pts})

df_all = pd.DataFrame(todas)

In [3]:
# Grandes clubes
GRANDES = ["Flamengo", "Palmeiras", "Corinthians", "Sao Paulo", "Santos",
           "Gremio", "Internacional", "Cruzeiro", "Atletico-MG",
           "Fluminense", "Vasco", "Botafogo"]

df_grandes = df_all[df_all["time"].isin(GRANDES)]

# Bump chart: posição final ao longo dos anos
fig = px.line(df_grandes, x="ano", y="posicao", color="time",
              title="Posição Final dos Grandes Clubes por Temporada<br><sup>Brasileirão Série A (2003-presente)</sup>",
              labels={"ano": "Ano", "posicao": "Posição Final", "time": "Clube"},
              markers=True)

fig.update_yaxes(autorange="reversed", dtick=2)  # Posição 1 no topo
fig.update_layout(
    template="plotly_white",
    width=1000,
    height=600,
    legend=dict(x=1.02, y=1),
    yaxis_title="Posição (1 = campeão)"
)

# Faixas de destaque
fig.add_hrect(y0=0.5, y1=4.5, fillcolor="green", opacity=0.05,
              annotation_text="Libertadores", annotation_position="left")
fig.add_hrect(y0=16.5, y1=20.5, fillcolor="red", opacity=0.05,
              annotation_text="Rebaixamento", annotation_position="left")

fig.show()
_path = "../charts/desempenho_clubes.html"
fig.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Série temporal da posição final dos grandes clubes, evidenciando a variância interanual e tendências de longo prazo."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/desempenho_clubes.html")

Gráfico exportado: charts/desempenho_clubes.html


In [4]:
# Ranking de títulos
campeoes = df_all[df_all["posicao"] == 1].groupby("time").size().sort_values(ascending=True)
campeoes = campeoes.reset_index()
campeoes.columns = ["time", "titulos"]

fig2 = px.bar(campeoes, x="titulos", y="time", orientation="h",
              title="Títulos do Brasileirão na Era Pontos Corridos<br><sup>2003-presente</sup>",
              labels={"titulos": "Títulos", "time": ""},
              color="titulos",
              color_continuous_scale="Greens",
              text="titulos")
fig2.update_layout(template="plotly_white", width=700, height=400, showlegend=False)
fig2.update_traces(textposition="outside")

fig2.show()
_path = "../charts/titulos.html"
fig2.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Distribuição de títulos na era dos pontos corridos com alta concentração: Palmeiras e Corinthians somam 38% dos campeonatos, caracterizando uma distribuição assimétrica com forte dominância de poucos clubes."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/titulos.html")

Gráfico exportado: charts/titulos.html


In [5]:
# Aproveitamento médio por clube (grandes)
aprov = df_grandes.copy()
# Aproveitamento = pontos / (jogos * 3)
# Cada temporada tem 38 rodadas = 38 jogos
aprov["aproveitamento"] = aprov["pontos"] / (38 * 3) * 100

media_aprov = aprov.groupby("time")["aproveitamento"].mean().sort_values(ascending=True).reset_index()

fig3 = px.bar(media_aprov, x="aproveitamento", y="time", orientation="h",
              title="Aproveitamento Médio dos Grandes Clubes<br><sup>Brasileirão (2003-presente)</sup>",
              labels={"aproveitamento": "Aproveitamento (%)", "time": ""},
              color="aproveitamento",
              color_continuous_scale="RdYlGn",
              text=media_aprov["aproveitamento"].round(1))
fig3.update_layout(template="plotly_white", width=800, height=500, showlegend=False)
fig3.update_traces(textposition="outside")

fig3.show()
_path = "../charts/aproveitamento_clubes.html"
fig3.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Ranking ordenado pela mediana do aproveitamento histórico, evidenciando a dispersão entre os grandes clubes. Palmeiras e Flamengo se destacam como outliers positivos na distribuição."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/aproveitamento_clubes.html")

Gráfico exportado: charts/aproveitamento_clubes.html
